# 18 — Chequeo del número de componentes PCA por bloque

Este cuaderno es exploratorio y diagnóstico: **no toca el cuaderno 17**. El objetivo es decidir, mirando solo varianza explicada y grados de libertad disponibles (nunca los resultados de H2/H3), qué número de componentes por bloque (dimensión × estadístico) es razonable usar como predictores en la regresión.

Contexto del problema: el cuaderno 17 usa `N_COMPONENTES = 3` de forma uniforme para los 12 bloques, pero la varianza que esos 3 componentes capturan varía muchísimo entre bloques (de ~26% a ~64%), porque el criterio de retención en el cuaderno 15 fue "≥70% de varianza acumulada", y eso implicó números de componentes muy distintos por bloque (de 5 a 21).

## 1. Carga de datos

**Objetivo:** cargar el archivo de varianza explicada por bloque (`15_varianza_explicada_por_bloque.xlsx`, exportado desde el cuaderno 15) y el de scores PCA, y fijar el N de países disponible para el modelo de regresión.

**Justificación metodológica:** todo el diagnóstico depende de estos dos archivos. Fijamos N=172 (universo tras excluir los 12 outliers monetarios de H1) porque es el mismo universo que usa el cuaderno 17 — la relación observaciones/predictores que evaluamos más abajo tiene que ser sobre ese N, no sobre los 184 o 193 países totales.

**Resultado esperado:** `df_varianza` (1.225 filas, 12 bloques) y `df_pca_scores` (184×173) cargados.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

IN_PATH_VARIANZA = Path("data/processed/15_varianza_explicada_por_bloque.xlsx")
IN_PATH_PCA = Path("data/processed/15_pca_scores.xlsx")

df_varianza = pd.read_excel(IN_PATH_VARIANZA)
df_pca_scores = pd.read_excel(IN_PATH_PCA, sheet_name="pca_scores", index_col=0)

N_PAISES_MODELO = 172  # tras excluir los 12 outliers monetarios (mismo universo que cuaderno 17)

DIMENSIONES = ["MONETARIA", "INSTITUCIONAL", "ESTRUCTURAL", "SOCIODEMOGRAFICA"]
ESTADISTICOS = ["media", "tendencia", "volatilidad"]

print(f"df_varianza: {df_varianza.shape} | bloques: {df_varianza['bloque'].nunique()}")
print(f"df_pca_scores: {df_pca_scores.shape}")
print(f"N países para el modelo de regresión: {N_PAISES_MODELO}")

df_varianza: (1225, 5) | bloques: 12
df_pca_scores: (184, 173)
N países para el modelo de regresión: 172


**Qué comprobar:** `df_varianza['bloque'].nunique() == 12` y `df_pca_scores.shape == (184, 173)`. Si tus rutas locales no son `data/processed/...`, ajustalas.

## 2. Varianza acumulada por bloque, para distintos k

**Objetivo:** para cada uno de los 12 bloques, calcular qué porcentaje de varianza queda capturado con los primeros k componentes (k=1 a 8), y armar una tabla comparativa.

**Justificación metodológica:** esta es la evidencia central del problema que planteaste — hay que verla en una sola tabla para comparar bloques directamente, en vez de mirar números sueltos.

**Resultado esperado:** `tabla_var_por_k`, con un bloque por fila y la varianza acumulada (%) en cada columna k=1..8.

In [2]:
K_MAX_EXPLORADO = 8

filas_var_por_k = []
for dim in DIMENSIONES:
    for est in ESTADISTICOS:
        bloque = f"{dim}_{est}"
        g = df_varianza[df_varianza["bloque"] == bloque].sort_values("componente")
        n_max = int(g["componente"].max())
        for k in range(1, K_MAX_EXPLORADO + 1):
            if k <= n_max:
                var_k = g.loc[g["componente"] == k, "var_acumulada"].values[0]
            else:
                var_k = g["var_acumulada"].max()  # el bloque no tiene mas componentes que k
            filas_var_por_k.append({
                "dimension": dim, "estadistico": est, "bloque": bloque,
                "k": k, "var_acumulada_pct": var_k * 100, "n_max_componentes": n_max,
            })

df_var_por_k = pd.DataFrame(filas_var_por_k)
tabla_var_por_k = df_var_por_k.pivot_table(index="bloque", columns="k", values="var_acumulada_pct").round(1)

pd.set_option("display.width", 200)
print(tabla_var_por_k)

k                                1     2     3     4     5     6     7     8
bloque                                                                      
ESTRUCTURAL_media             18.1  26.8  34.1  39.0  43.1  46.7  50.0  53.0
ESTRUCTURAL_tendencia         10.6  19.7  26.3  31.5  36.1  40.3  44.1  47.4
ESTRUCTURAL_volatilidad       19.8  30.4  36.1  40.4  43.9  46.9  49.6  52.1
INSTITUCIONAL_media           50.6  59.2  63.7  67.2  70.3  72.7  75.0  77.0
INSTITUCIONAL_tendencia       26.6  36.2  43.8  48.6  52.2  55.3  58.2  60.6
INSTITUCIONAL_volatilidad     30.1  37.1  42.1  46.5  50.0  53.4  56.6  59.3
MONETARIA_media               17.9  28.4  37.5  43.4  48.5  52.9  57.3  61.6
MONETARIA_tendencia           12.1  20.9  28.3  34.8  40.2  45.4  50.3  55.0
MONETARIA_volatilidad         19.2  28.2  34.4  39.8  45.1  50.1  54.6  58.7
SOCIODEMOGRAFICA_media        30.3  40.7  48.4  53.3  58.2  61.4  64.4  66.8
SOCIODEMOGRAFICA_tendencia    17.4  23.3  29.0  33.7  37.7  41.2  44.5  47.3

**Qué comprobar:** en la fila `INSTITUCIONAL_media`, la columna `k=3` debería dar ~63.7%, y en `ESTRUCTURAL_tendencia`, ~26.3% — son los números que ya habías detectado a mano. Si no coinciden, algo cambió en el archivo de varianza respecto de lo que revisamos.

**Cómo leerlo:** cuanto más rápido sube una fila, menos componentes necesita ese bloque para capturar su varianza — MONETARIA y ESTRUCTURAL suben lento (variables más heterogéneas entre sí); INSTITUCIONAL y SOCIODEMOGRÁFICA_media suben rápido (variables más redundantes entre sí, coherente con sus KMO más altos).

## 3. Curvas de varianza acumulada (scree plot, los 12 bloques)

**Objetivo:** visualizar la tabla anterior como curvas, extendiendo hasta el máximo de componentes de cada bloque, para ver dónde se aplana cada una.

**Justificación metodológica:** el "codo" de cada curva es el punto donde agregar más componentes deja de aportar mucho — es la referencia visual estándar para decisiones de este tipo (aunque acá la decisión final también depende de grados de libertad, no solo de este gráfico).

**Resultado esperado:** un gráfico con 12 curvas y líneas de referencia en 30/40/50/60/70% de varianza.

In [3]:
K_LIMITE_GRAFICO = 25

fig_scree = go.Figure()
for dim in DIMENSIONES:
    for est in ESTADISTICOS:
        bloque = f"{dim}_{est}"
        g = df_varianza[df_varianza["bloque"] == bloque].sort_values("componente")
        g = g[g["componente"] <= K_LIMITE_GRAFICO]
        fig_scree.add_trace(go.Scatter(
            x=g["componente"], y=g["var_acumulada"] * 100,
            mode="lines+markers", name=bloque,
            line=dict(width=2), marker=dict(size=4),
        ))

for umbral in [30, 40, 50, 60, 70]:
    fig_scree.add_hline(y=umbral, line_dash="dot", line_color="gray",
                         annotation_text=f"{umbral}%", annotation_position="right")

fig_scree.update_layout(
    title="Varianza acumulada por componente — los 12 bloques (dimensión × estadístico)",
    xaxis_title="Componente (k)", yaxis_title="Varianza acumulada (%)",
    height=600, legend=dict(font=dict(size=9)),
)
fig_scree.show()

if "figuras_reporte" not in dir():
    figuras_reporte = {}
figuras_reporte["varianza_acumulada_scree_12_bloques"] = fig_scree

**Qué comprobar:** deberían verse 12 líneas, todas empezando en algún punto entre 10% y 50% (k=1) y subiendo hasta cruzar las líneas de referencia en distintos k. Las curvas de MONETARIA, ESTRUCTURAL y SOCIODEMOGRAFICA_tendencia/volatilidad son las que suben más lento — son las que generan el problema que detectaste.

## 4. Componentes necesarios por bloque, según umbral de varianza objetivo

**Objetivo:** para cada bloque, calcular el k mínimo necesario para alcanzar distintos umbrales de varianza (30%, 40%, 50%, 60%, 70%).

**Justificación metodológica:** esto traduce las curvas del punto 3 en números concretos y comparables, que vamos a cruzar con grados de libertad en el punto 5.

**Resultado esperado:** `df_k_por_umbral`, con una fila por bloque y el k necesario en cada columna de umbral.

In [4]:
UMBRALES = [0.30, 0.40, 0.50, 0.60, 0.70]

filas_umbral = []
for dim in DIMENSIONES:
    for est in ESTADISTICOS:
        bloque = f"{dim}_{est}"
        g = df_varianza[df_varianza["bloque"] == bloque].sort_values("componente")
        fila = {"dimension": dim, "estadistico": est, "bloque": bloque,
                "n_max_componentes": int(g["componente"].max())}
        for u in UMBRALES:
            sub = g[g["var_acumulada"] >= u]
            fila[f"k_para_{int(u*100)}pct"] = int(sub["componente"].min()) if len(sub) else None
        filas_umbral.append(fila)

df_k_por_umbral = pd.DataFrame(filas_umbral)
pd.set_option("display.width", 220)
print(df_k_por_umbral.drop(columns="bloque").to_string(index=False))

       dimension estadistico  n_max_componentes  k_para_30pct  k_para_40pct  k_para_50pct  k_para_60pct  k_para_70pct
       MONETARIA       media                 63             3             4             6             8            11
       MONETARIA   tendencia                 60             4             5             7            10            13
       MONETARIA volatilidad                 63             3             5             6             9            12
   INSTITUCIONAL       media                 80             1             1             1             3             5
   INSTITUCIONAL   tendencia                 80             2             3             5             8            13
   INSTITUCIONAL volatilidad                 80             1             3             5             9            14
     ESTRUCTURAL       media                120             3             5             7            11            16
     ESTRUCTURAL   tendencia                118         

**Qué comprobar:** `INSTITUCIONAL_media` debería necesitar muy pocos componentes incluso para 60-70% (bloque muy redundante, KMO=0.92), mientras que `ESTRUCTURAL_tendencia` y `SOCIODEMOGRAFICA_tendencia` necesitan muchos (19-21) para llegar a 70% — coherente con la tabla del cuaderno 15.

## 5. Factibilidad: grados de libertad para la regresión del cuaderno 17

**Objetivo:** para cada estadístico (media, tendencia, volatilidad) — que es donde el cuaderno 17 corre una regresión separada con las 4 dimensiones como predictores — calcular cuántos predictores implicaría cada umbral de varianza, bajo dos escenarios: **k uniforme** (mismo k en las 4 dimensiones, igual al peor caso) y **k asimétrico** (el k mínimo de cada dimensión, sumado).

**Justificación metodológica:** el cuaderno 17 no corre un único modelo con 36 predictores — corre un modelo por estadístico, con 4 dimensiones × k componentes cada uno (con k=3 uniforme, eso son 12 predictores por regresión, no 36). El N disponible por estadístico (167 para media, 166 para tendencia y volatilidad) ya está documentado en la salida de H3 del cuaderno 17 — no lo recalculamos acá para no duplicar esa lógica.

Como regla práctica para no "reventar" la regresión (tu palabra, y es la correcta): apuntamos a mantener **al menos ~10 observaciones por predictor**. Es un umbral convencional, no una ley matemática — algunos autores piden más (15-20), pero por debajo de 10 el riesgo de sobreajuste y coeficientes inestables ya es alto en OLS estándar.

**Resultado esperado:** `df_factibilidad`, con el ratio observaciones/predictor para cada combinación de estadístico × umbral × escenario, y un flag de si cumple el mínimo de 10.

In [5]:
# N disponible por estadistico: documentado en el cuaderno 17 (seccion H3),
# ya excluidos los 12 outliers monetarios y las filas con NaN en el target de inflacion.
# Si el cuaderno 17 se vuelve a correr y estos numeros cambian, actualizar aca.
N_DISPONIBLE = {"media": 167, "tendencia": 166, "volatilidad": 166}
RATIO_MINIMO_RECOMENDADO = 10  # regla practica habitual: >=10 observaciones por predictor en OLS

filas_factibilidad = []
for est in ESTADISTICOS:
    sub = df_k_por_umbral[df_k_por_umbral["estadistico"] == est]
    n = N_DISPONIBLE[est]
    for u in UMBRALES:
        col = f"k_para_{int(u*100)}pct"
        pred_asimetrico = int(sub[col].sum())
        k_uniforme = int(sub[col].max())
        pred_uniforme = 4 * k_uniforme
        filas_factibilidad.append({
            "estadistico": est, "umbral_varianza_pct": int(u * 100), "N_disponible": n,
            "k_uniforme_necesario": k_uniforme, "predictores_uniforme": pred_uniforme,
            "ratio_obs_predictor_uniforme": round(n / pred_uniforme, 1),
            "predictores_asimetrico": pred_asimetrico,
            "ratio_obs_predictor_asimetrico": round(n / pred_asimetrico, 1),
        })

df_factibilidad = pd.DataFrame(filas_factibilidad)
df_factibilidad["uniforme_factible"] = df_factibilidad["ratio_obs_predictor_uniforme"] >= RATIO_MINIMO_RECOMENDADO
df_factibilidad["asimetrico_factible"] = df_factibilidad["ratio_obs_predictor_asimetrico"] >= RATIO_MINIMO_RECOMENDADO

pd.set_option("display.width", 220)
print(df_factibilidad.to_string(index=False))

if "tablas_reporte" not in dir():
    tablas_reporte = {}
tablas_reporte["factibilidad_gl_h2_h3"] = df_factibilidad

estadistico  umbral_varianza_pct  N_disponible  k_uniforme_necesario  predictores_uniforme  ratio_obs_predictor_uniforme  predictores_asimetrico  ratio_obs_predictor_asimetrico  uniforme_factible  asimetrico_factible
      media                   30           167                     3                    12                          13.9                       8                            20.9               True                 True
      media                   40           167                     5                    20                           8.3                      12                            13.9              False                 True
      media                   50           167                     7                    28                           6.0                      18                             9.3              False                False
      media                   60           167                    11                    44                           3.8            

**Qué comprobar y cómo leerlo:**
- Con **k=3 uniforme (el diseño actual del cuaderno 17)**, el ratio obs/predictor está entre ~13 y ~14 — eso está bien en términos de grados de libertad. El problema del cuaderno 17 **no es de grados de libertad, es de varianza capturada de forma desigual**.
- Al umbral de **30%**, tanto el escenario uniforme como el asimétrico siguen siendo factibles (ratio ≥10) en los tres estadísticos.
- A partir de **40%**, el escenario uniforme deja de ser factible en los tres estadísticos, y el asimétrico también deja de serlo en tendencia y volatilidad (ratio ~8.3-9.8, por debajo de 10).
- A partir de **50%**, ninguno de los dos escenarios es factible en ningún estadístico.

Esto confirma numéricamente lo que sospechábamos: **no existe un k que capture 50%+ de varianza en todos los bloques sin comprometer los grados de libertad.** El margen real de maniobra está entre 30% y 40%.

## 6. Configuración candidata (k asimétrico, umbral 30%)

**Objetivo:** generar la tabla de k recomendado por bloque, usando el umbral de 30% (el más alto que resulta factible en los tres estadísticos según el punto 5), y exportarla para uso posterior en el cuaderno 17.

**Justificación metodológica:** 30% es una recomendación técnica basada exclusivamente en varianza y grados de libertad — no se miró ningún resultado de H2/H3 para llegar a este número. Es una configuración *candidata*, no una decisión cerrada: hay bloques que con esto capturan bastante más de 30% (INSTITUCIONAL_media, 50.6% con k=1) y otros que quedan justo en el piso (ESTRUCTURAL_tendencia, 31.5% con k=4) — sigue habiendo asimetría, pero mucho más acotada que con k=3 uniforme (que iba de 26% a 64%).

**Resultado esperado:** `df_k_recomendado` y el archivo `18_k_recomendado_por_bloque.xlsx`.

In [6]:
UMBRAL_ELEGIDO = 0.30  # ver justificacion arriba y en la conclusion

filas_recomendado = []
for dim in DIMENSIONES:
    for est in ESTADISTICOS:
        bloque = f"{dim}_{est}"
        g = df_varianza[df_varianza["bloque"] == bloque].sort_values("componente")
        sub = g[g["var_acumulada"] >= UMBRAL_ELEGIDO]
        k_rec = int(sub["componente"].min()) if len(sub) else int(g["componente"].max())
        var_en_k = float(g.loc[g["componente"] == k_rec, "var_acumulada"].values[0])
        filas_recomendado.append({
            "dimension": dim, "estadistico": est, "bloque": bloque,
            "k_recomendado": k_rec, "var_acumulada_pct": round(var_en_k * 100, 1),
        })

df_k_recomendado = pd.DataFrame(filas_recomendado)
print(df_k_recomendado.to_string(index=False))

print("\nPredictores totales por regresion (suma de k, 4 dimensiones), por estadistico:")
print(df_k_recomendado.groupby("estadistico")["k_recomendado"].sum())

if "tablas_reporte" not in dir():
    tablas_reporte = {}
tablas_reporte["k_recomendado_por_bloque"] = df_k_recomendado

OUT_PATH = Path("data/processed/18_k_recomendado_por_bloque.xlsx")
df_k_recomendado.to_excel(OUT_PATH, index=False)
print(f"\nExportado: {OUT_PATH}")

       dimension estadistico                       bloque  k_recomendado  var_acumulada_pct
       MONETARIA       media              MONETARIA_media              3               37.5
       MONETARIA   tendencia          MONETARIA_tendencia              4               34.8
       MONETARIA volatilidad        MONETARIA_volatilidad              3               34.4
   INSTITUCIONAL       media          INSTITUCIONAL_media              1               50.6
   INSTITUCIONAL   tendencia      INSTITUCIONAL_tendencia              2               36.2
   INSTITUCIONAL volatilidad    INSTITUCIONAL_volatilidad              1               30.1
     ESTRUCTURAL       media            ESTRUCTURAL_media              3               34.1
     ESTRUCTURAL   tendencia        ESTRUCTURAL_tendencia              4               31.5
     ESTRUCTURAL volatilidad      ESTRUCTURAL_volatilidad              2               30.4
SOCIODEMOGRAFICA       media       SOCIODEMOGRAFICA_media              1        

**Qué comprobar:** la suma de `k_recomendado` por estadístico debería dar 8 (media), 14 (tendencia) y 9 (volatilidad) — con ratios obs/predictor de ~20.9, ~11.9 y ~18.4 respectivamente (los mismos números de la tabla del punto 5, fila 30%).

## 7. Conclusión

**Hechos (de esta exploración, no opinables):**
- Con el criterio actual del cuaderno 17 (k=3 uniforme), la varianza capturada por bloque va de 26.3% (ESTRUCTURAL_tendencia) a 63.7% (INSTITUCIONAL_media) — una diferencia de más de 2,4x.
- No existe ningún k (uniforme ni asimétrico) que capture ≥50% de varianza en los 12 bloques sin bajar el ratio observaciones/predictor por debajo de 10 en al menos un estadístico.
- Un umbral de 30% de varianza es el techo factible en los tres estadísticos, tanto en escenario uniforme como asimétrico.

**Mi recomendación metodológica:** pasar de "3 componentes fijos por bloque" a "k asimétrico por bloque, definido por un piso de varianza explicada (30%)". Esto reduce la asimetría entre bloques (de un rango de 26-64% a uno más parejo, todos ≥30%) sin empeorar los grados de libertad frente al diseño actual — de hecho, en `media` mejora el ratio (20.9 vs 13.9 actual) porque varios bloques necesitan menos de 3 componentes para llegar a 30%.

**Esto es mi criterio, no un hecho:** otra persona podría preferir mantener la simetría de "mismo k en todos los bloques" por razones de comparabilidad interpretativa entre dimensiones (más fácil decir "cada dimensión aporta 3 componentes" que explicar por qué una aporta 1 y otra 4). Es una decisión legítima, con el costo de que la varianza capturada seguirá siendo desigual. Te dejo la decisión abierta — con `df_k_recomendado` ya armado, se puede ajustar el umbral del punto 6 fácilmente antes de decidir.

**Lo que sigue:** una vez que confirmes el criterio (asimétrico a 30%, otro umbral, o volver a uniforme con su asimetría conocida), ajustamos la sección 3 del cuaderno 17 para que lea `18_k_recomendado_por_bloque.xlsx` en vez de aplicar `N_COMPONENTES=3` fijo — y ahí sí, recién ahí, corremos H2/H3 con la configuración cerrada.